In [2]:
import os
import pandas as pd
from sumy.parsers.plaintext import PlaintextParser
from sumy.nlp.tokenizers import Tokenizer
from sumy.summarizers.lex_rank import LexRankSummarizer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch

/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [12]:
# === Настройки моделей ===
SUMY_LANG = 'russian'
T5_MODEL = 'IlyaGusev/rut5_base_sum_gazeta'

In [13]:
# === Загрузка корпуса из локальной папки ===
def load_texts_2024(base_folder: str) -> pd.DataFrame:
    """
    Обходит только папку IMS2024 и собирает статьи .txt
    с именами вида *_IMS_2024_rus.txt.
    Возвращает DataFrame с колонками ['name','text','year'] (year='2024').
    """
    records = []
    ims2024_dir = os.path.join(base_folder, 'IMS2024')
    if not os.path.isdir(ims2024_dir):
        raise FileNotFoundError(f"Папка {ims2024_dir} не найдена")
    for fname in os.listdir(ims2024_dir):
        if fname.endswith('_IMS_2024_rus.txt'):
            path = os.path.join(ims2024_dir, fname)
            with open(path, 'r', encoding='utf-8', errors='ignore') as f:
                text = f.read().strip()
            name = os.path.splitext(fname)[0]
            records.append({'name': name, 'text': text, 'year': '2024'})
    return pd.DataFrame(records)


In [14]:
base_folder = '/Users/juliak/Downloads/IMS2013-20242'
df_2024 = load_texts_2024(base_folder)
print("Статей за 2024:", len(df_2024))

Статей за 2024: 5


In [15]:
# === Функция суммаризации Sumy (LexRank) ===
def summarize_sumy(text: str, sentences_count: int = 3) -> str:
    """
    Возвращает первые sentences_count предложений LexRank-резюме для текста.
    """
    parser = PlaintextParser.from_string(text, Tokenizer(SUMY_LANG))
    summarizer = LexRankSummarizer()
    summary = summarizer(parser.document, sentences_count)
    return ' '.join(str(s) for s in summary)

In [16]:
# === Функция суммаризации T5 ===
# Загрузка модели один раз
_t5_tokenizer = AutoTokenizer.from_pretrained(T5_MODEL)
_t5_model = AutoModelForSeq2SeqLM.from_pretrained(T5_MODEL)

def summarize_t5(text: str,
                 max_length: int = 200,
                 min_length: int = 50,
                 num_beams: int = 4) -> str:
    """
    Генерирует реферат с помощью T5-модели.
    Truncates input to 512 токенов.
    """
    inputs = _t5_tokenizer(
        text,
        max_length=512,
        truncation=True,
        return_tensors='pt'
    )
    out = _t5_model.generate(
        input_ids=inputs.input_ids,
        attention_mask=inputs.attention_mask,
        max_length=max_length,
        min_length=min_length,
        num_beams=num_beams,
        no_repeat_ngram_size=3
    )
    return _t5_tokenizer.decode(out[0], skip_special_tokens=True)

In [18]:
# === Пример использования ===
if __name__ == '__main__':
    # загрузка небольшого примера
    folder = '/Users/juliak/Downloads/IMS2013-20242'
    df = load_texts_2024(folder)
    print("Loaded texts:", len(df))
    sample = df.text.iloc[0]
    print("\nOriginal (first 300 chars):\n", sample[:300], "...\n")

Loaded texts: 5

Original (first 300 chars):
 Под  тематическим  моделированием  традиционно  понимается  особый  способ построения  структурно-семантической  модели  корпуса  текстов,  которая  определяет взаимосвязи тем, документов и слов-тематизаторов [3]. Темы рассматриваются как скрытые факторы,  представленные  кластерами  слов-тематизато ...



In [19]:
    # Sumy
    print("Sumy LexRank summary (3 sentences):")
    print(summarize_sumy(sample, sentences_count=3))

Sumy LexRank summary (3 sentences):
В статье представлены результаты построения тематических моделей корпуса ТКиКЛ с помощью  алгоритмов  NMF,  LSA,  LDA  и  Biterm. Результаты тематического моделирования корпуса ТКиКЛ2.1. Результаты генерации меток тем в корпусе ТКиКЛ3.1.


In [20]:
    # T5
    print("\nT5 summary (50-150 tokens):")
    print(summarize_t5(sample, min_length=50, max_length=150))


T5 summary (50-150 tokens):
Темы, документы и слова-тематизаторы могут пересекаться в одном или нескольких темах с некоторой вероятностью. Это позволит объективно оценить и сопоставить результаты построения тематических моделей корпуса ТКиКЛ с помощью алгоритмов Biterm, LSA, LDA и Biterm.


In [21]:
    # Можно расширить DataFrame
    df['sumy'] = df.text.apply(lambda t: summarize_sumy(t, sentences_count=2))
    df['t5'] = df.text.apply(lambda t: summarize_t5(t, min_length=30, max_length=100))

In [23]:
df

,name,text,year,sumy,t5
0,MitrofanovaGolubev_IMS_2024_rus,Под тематическим моделированием традиционно...,2024,В статье представлены результаты построения те...,"Темы, документы и слова-тематизаторы могут пер..."
1,MitrofanovaAdamova_IMS_2024_rus,"1. Введение Проект, представленный в данной ст...",2024,В данной статье рассматриваются следующие ...,В Петербургской школе корпусной и компьютерной...
2,Sukhan_IMS_2024_rus,1. Введение: типы метаинформации в корпусе и н...,2024,"Наконец, ресурс не позволяет изменять цвет, ра...","Корпус статей по корпусной лингвистике, собран..."
3,Vybornaya_IMS_2024_rus,1. ВведениеДанная статья основана на результат...,2024,"При этом как каузатив, так и квалификатив ...",Предлоги и предложные конструкции могут быть п...
4,Belkin_IMS_2024_rus,1. Введение По мере стремительного развития...,2024,Существует множество традиционных и совреме...,Введение пользовательских отзывов может помочь...
